# Répartition gestion directe / gestion partagée par bénéficiaire
### Traitement Python → fichier unique à importer dans MicroStrategy

Produit **un seul fichier** dénormalisé, prêt à importer dans l'espace data viz
de MicroStrategy : aucune liaison à créer, aucun risque de multiplication de lignes.

**Grain** : une ligne par **SIREN × année**, avec les deux montants côte à côte.

**Colonnes produites**

| Colonne | Contenu |
|---|---|
| `SIREN` | clé de rapprochement (texte, zéros conservés) |
| `NOM_REFERENCE` | nom officiel INSEE (`Nom_API`) — voir règle ci-dessous |
| `SOURCE_NOM_REFERENCE` | traçabilité : SIRENE / plus fréquent / unique |
| `NOM_FTS`, `NOM_KOHESIO` | nom tel qu'il apparaît dans **chaque** base |
| `ANNEE` | exercice FTS / année de début de financement Kohesio |
| `MONTANT_FTS`, `MONTANT_KOHESIO` | montants **séparés** par canal |
| `MONTANT_TOTAL_INDICATIF` | somme — **indicative**, voir note |
| `PART_DIRECTE_ANNEE`, `PART_PARTAGEE_ANNEE` | **répartition de l'entité cette année-là** |
| `PART_DIRECTE_PERIODE`, `PART_PARTAGEE_PERIODE` | **répartition sur toute la période** (plus robuste) |
| `PRESENCE_ANNEE`, `PRESENCE_PERIODE` | FTS seul / Kohesio seul / Les deux |
| SGAE, NAF, section, opérateur d'État | attributs d'entité pour filtrer |

> **Règle du nom de référence** : `Nom_API` en priorité (nom légal SIRENE,
> identique dans les deux bases car produit par le même moteur), sinon le nom le
> plus fréquent, sinon l'unique valeur. La colonne `SOURCE_NOM_REFERENCE` indique
> laquelle a servi — les noms certifiés sont ainsi distingués des conventions.

> **⚠️ Le total est indicatif.** Le FTS porte des engagements annuels, Kohesio un
> montant programmé sur toute la durée du projet, rattaché à son année de
> démarrage. Le total sert à **hiérarchiser** les acteurs, pas à produire un
> montant comptable. La répartition **sur la période** est plus fiable que
> l'annuelle, qui est sensible aux à-coups de calendrier.


## 0 · Paramètres

In [ ]:
!pip -q install pandas openpyxl >/dev/null 2>&1
import pandas as pd, numpy as np, re

FICHIER_SORTIE = "REPARTITION_CANAUX_MICROSTRATEGY.xlsx"

FORMAT         = "xlsx"        # "xlsx" ou "csv"

# ── KOHESIO : montant = PART UE du projet (assiette homogène avec le FTS) ───
KOH = {"siren":"SIREN", "nom":"Beneficiary_Name",
       "montant":"Project_EU_Budget",
       "annee":"Annee", "date":"Operation_Start_Date",
       "projet":"Operation_Unique_Identifier"}

# ── FTS : montant = montant contractualisé avec le bénéficiaire ─────────────
FTS = {"siren":"SIREN", "nom":None,
       "montant":"Beneficiary contracted amount",
       "annee":None, "projet":None}

# ⚠️ ALTERNATIVE — pour mesurer l'INVESTISSEMENT TOTAL mobilisé (cofinancement
#    national compris) plutôt que la seule contribution européenne, décommenter :
# KOH["montant"] = "Total_Eligible_Expenditure_amount"
#
# Assiette retenue : les DEUX canaux mesurent désormais une contribution
# européenne (part UE du projet côté Kohesio, montant contractualisé côté FTS).
# C'est le choix le plus cohérent pour comparer ce que chaque acteur capte
# de l'Europe. Ordre de grandeur : sur un projet à 511 821 € de dépense
# éligible, la part UE n'est que de 307 093 € — l'écart est important.

# ── Colonne du NOM OFFICIEL SIRENE (celle qui sert de NOM_API / nom unique) ──
# None = détection automatique. Renseignez le nom EXACT si la détection échoue.
COL_NOM_API_FTS = "Nom API"   # ← nom EXACT dans le fichier FTS (avec espace)
COL_NOM_API_KOH = "Nom_API"   # ← nom dans le fichier Kohesio (avec underscore)

# Montants DÉJÀ AGRÉGÉS par bénéficiaire : jamais sommables
MONTANTS_INTERDITS = ("montant_total_beneficiaire", "total_beneficiaire")

# Attributs d'entité repris dans le fichier final
ATTRIBUTS = ["Référentiel SGAE","Activite_principale","Section_NAF","Code_NAF_APE",
             "Forme_juridique","Etat_entreprise","Operateur_Etat","Programme_Operateur"]


## 1 · Chargement

In [ ]:
def _charger(libelle, defaut):
    try:
        from google.colab import files as _cf
        print(f"Déposez le fichier {libelle} :")
        up = _cf.upload()
        return pd.read_excel(list(up.keys())[0], dtype=str)
    except Exception:
        return pd.read_excel(defaut, dtype=str)

df_fts = _charger("FTS enrichi (avec SIREN)",     "FTS_enrichi.xlsx")
df_koh = _charger("KOHESIO enrichi (avec SIREN)", "KOHESIO_beneficiaires_siretise.xlsx")
print(f"FTS : {len(df_fts)} lignes | KOHESIO : {len(df_koh)} lignes\n")

for _l,_d in (("FTS",df_fts),("KOHESIO",df_koh)):
    print(f"── Colonnes {_l} ({len(_d.columns)}) ──")
    print("   " + " | ".join(map(str,_d.columns)) + "\n")

if list(df_fts.columns) == list(df_koh.columns):
    print("⚠️  Les deux fichiers ont les mêmes colonnes : vous avez sans doute "
          "chargé deux fois le même fichier.")


## 2 · Normalisation et mise au format long

Le notebook **affiche les colonnes retenues** pour chaque base. Si la détection
FTS se trompe, corrigez le dictionnaire `FTS` en cellule 0 et relancez.

In [ ]:
def trouver(cols,*mots,exclure=()):
    for m in mots:
        for c in cols:
            cl=str(c).lower()
            if m.lower() in cl and not any(x.lower() in cl for x in exclure): return c
    return None

def norm_siren(s):
    s=re.sub(r"\D","",str(s or ""))
    return s.zfill(9) if s and len(s)<=9 else (s[:9] if s else "")

def to_num(s, libelle=""):
    """Conversion numérique robuste.

    Règle de décision (sans ambiguïté) :
      - virgule ET point  → le DERNIER des deux est le séparateur décimal
      - virgule SEULE     → séparateur DÉCIMAL (convention française), sauf si
                            elle est suivie d'exactement 3 chiffres ET répétée
                            ou précédée d'un groupe de milliers cohérent
      - point SEUL        → séparateur décimal

    ⚠️ Une virgule seule n'est traitée comme séparateur de milliers QUE si le
    motif est celui d'un nombre anglo-saxon complet (ex. 1,234,567), c'est-à-dire
    plusieurs virgules, ou groupes de 3 chiffres réguliers. Sinon, on privilégie
    l'interprétation décimale : mieux vaut sous-estimer que multiplier par 1000.
    """
    t = (s.astype(str)
           .str.replace(r"[\s\u00a0\u202f]", "", regex=True)
           .str.replace(r"[€$£]|EUR|eur", "", regex=True)
           .str.strip())
    t = t.replace({"": np.nan, "nan": np.nan, "None": np.nan, "-": np.nan})

    _MILLIERS = re.compile(r"^\d{1,3}(,\d{3})+$")     # 1,234  1,234,567

    def _conv(x):
        if not isinstance(x, str) or not x:
            return np.nan
        neg = x.startswith("(") and x.endswith(")")
        if neg: x = x[1:-1]
        a, b = x.rfind(","), x.rfind(".")
        if a > -1 and b > -1:
            if b > a: x = x.replace(",", "")                       # 1,234.56 (EN)
            else:     x = x.replace(".", "").replace(",", ".")     # 1.234,56 (FR)
        elif a > -1:
            # virgule SEULE : milliers uniquement si motif anglo-saxon strict
            x = x.replace(",", "") if _MILLIERS.match(x) else x.replace(",", ".")
        try:
            v = float(x)
            return -v if neg else v
        except ValueError:
            return np.nan

    out = t.map(_conv)
    n_ko = out.isna().sum() - s.isna().sum()
    if n_ko > 0 and libelle:
        ex = s[out.isna() & s.notna()].astype(str).head(3).tolist()
        print(f"   ⚠️ [{libelle}] {n_ko} montant(s) non convertis — ex. : {ex}")
    return out

def preparer(df, canal, mapping):
    cols=df.columns.tolist()
    def pick(cle,*auto,exclure=()):
        v=mapping.get(cle)
        if v and v in cols: return v
        if v and v not in cols: print(f"  ⚠️ [{canal}] {v!r} absente → détection auto")
        return trouver(cols,*auto,exclure=exclure)

    c_siren=pick("siren","siren")
    c_nom  =pick("nom","beneficiary_name","beneficiarylabel","name of beneficiary","nom",
                 exclure=("api","unique","refer"))
    _forced = COL_NOM_API_FTS if canal=="Gestion directe" else COL_NOM_API_KOH
    c_api  = (_forced if (_forced and _forced in cols)
              else trouver(cols,"nom_api","nom api","nom_sirene","nom sirene",
                           "raison_sociale","raison sociale","nom_complet","nom complet",
                           "nom unique","nom_unique","denomination"))
    c_mont =pick("montant","project_eu_budget","commitment total","montant total",
                 "montant","amount","budget",exclure=MONTANTS_INTERDITS)
    c_an   =mapping.get("annee") if mapping.get("annee") in cols else \
            (next((c for c in cols if str(c).strip().lower() in ("annee","année","year")),None))
    c_date =mapping.get("date") if mapping.get("date") in cols else \
            trouver(cols,"operation_start_date","start date","date de début")
    c_proj =pick("projet","operation_unique_identifier","reference of the legal","projet")

    if not c_siren: raise KeyError(f"[{canal}] SIREN introuvable parmi {cols[:12]}…")
    if c_mont and any(x in str(c_mont).lower() for x in MONTANTS_INTERDITS):
        raise ValueError(f"[{canal}] {c_mont!r} est un montant déjà agrégé : non sommable.")

    o=pd.DataFrame(index=df.index)
    o["SIREN"]=df[c_siren].map(norm_siren)
    o["CANAL"]=canal
    o["MONTANT"]=to_num(df[c_mont], f"{canal}/{c_mont}") if c_mont else np.nan
    o["NOM"]=df[c_nom].astype(str) if c_nom else ""
    o["NOM_API"]=df[c_api].astype(str) if c_api else ""
    o["PROJET"]=df[c_proj].astype(str) if c_proj else ""
    for a in ATTRIBUTS: o[a]=df[a] if a in df.columns else np.nan

    if c_an:
        o["ANNEE"]=pd.to_numeric(df[c_an].astype(str).str.extract(r"(\d{4})")[0],errors="coerce")
        org=f"colonne « {c_an} »"
    elif c_date:
        o["ANNEE"]=pd.to_datetime(df[c_date],dayfirst=True,errors="coerce").dt.year
        org=f"dérivée de « {c_date} »"
    else:
        o["ANNEE"]=np.nan; org="AUCUNE"

    _tot = o["MONTANT"].sum()
    print(f"[{canal}] montant total converti : {_tot:,.0f} €".replace(","," "))
    ok=o["SIREN"].str.len().eq(9) & o["ANNEE"].notna()
    print(f"[{canal}] siren={c_siren!r} nom={c_nom!r} montant={c_mont!r} année={org}")
    if c_api:
        print(f"[{canal}] nom officiel SIRENE = {c_api!r} "
              f"({o['NOM_API'].astype(str).str.strip().ne('').sum()}/{len(o)} renseignés)")
    else:
        print(f"[{canal}] ⚠️ AUCUNE colonne de nom officiel SIRENE trouvée — "
              f"le nom brut servira de repli.\n"
              f"        Colonnes disponibles : {', '.join(map(str,cols[:25]))}…\n"
              f"        → renseignez COL_NOM_API_{'FTS' if canal=='Gestion directe' else 'KOH'} en cellule 0.")
    print(f"[{canal}] {ok.sum()}/{len(o)} lignes retenues "
          f"({(~o['SIREN'].str.len().eq(9)).sum()} sans SIREN, {o['ANNEE'].isna().sum()} sans année)")
    return o[ok].copy()

lg_fts=preparer(df_fts,"Gestion directe", FTS)
lg_koh=preparer(df_koh,"Gestion partagée",KOH)
empile=pd.concat([lg_fts,lg_koh],ignore_index=True)
print(f"\nTotal : {len(empile)} lignes | {empile['SIREN'].nunique()} SIREN distincts")


## 3 · Nom de référence (priorité `Nom_API`) et noms par base

In [ ]:
def _plus_freq(s):
    s=s[s.astype(str).str.strip().ne("") & s.notna() & s.astype(str).ne("nan")]
    return s.value_counts().index[0] if len(s) else ""

ref=[]
for siren,g in empile.groupby("SIREN"):
    api=_plus_freq(g["NOM_API"])
    if api:
        nom,src=api,"SIRENE (Nom_API)"
    else:
        noms=g["NOM"][g["NOM"].astype(str).str.strip().ne("")]
        if noms.nunique()==1: nom,src=noms.iloc[0],"valeur unique"
        elif len(noms):       nom,src=_plus_freq(noms),"nom le plus fréquent"
        else:                 nom,src="(inconnu)","aucun nom"
    ref.append((siren,nom,src))
lu_nom=pd.DataFrame(ref,columns=["SIREN","NOM_REFERENCE","SOURCE_NOM_REFERENCE"])

# nom tel qu'il apparaît dans chaque base
def noms_par_canal(canal,col):
    d=empile[empile["CANAL"]==canal]
    return (d[d["NOM"].astype(str).str.strip().ne("")]
            .groupby("SIREN")["NOM"].apply(lambda s:" | ".join(sorted(set(s)))).rename(col))
lu_nom=(lu_nom.merge(noms_par_canal("Gestion directe","NOM_FTS"),on="SIREN",how="left")
              .merge(noms_par_canal("Gestion partagée","NOM_KOHESIO"),on="SIREN",how="left"))
lu_nom[["NOM_FTS","NOM_KOHESIO"]]=lu_nom[["NOM_FTS","NOM_KOHESIO"]].fillna("")

print(lu_nom["SOURCE_NOM_REFERENCE"].value_counts().to_string())
print(f"\nEntités nommées différemment dans les deux bases : "
      f"{((lu_nom['NOM_FTS'].ne('')) & (lu_nom['NOM_KOHESIO'].ne('')) & (lu_nom['NOM_FTS']!=lu_nom['NOM_KOHESIO'])).sum()}")


## 4 · Croisement SIREN × année et calcul des répartitions

In [ ]:
piv=(empile.assign(ANNEE=empile["ANNEE"].astype(int))
      .pivot_table(index=["SIREN","ANNEE"],columns="CANAL",values="MONTANT",
                   aggfunc="sum",fill_value=0).reset_index())
for c in ("Gestion directe","Gestion partagée"):
    if c not in piv.columns: piv[c]=0.0
piv=piv.rename(columns={"Gestion directe":"MONTANT_FTS","Gestion partagée":"MONTANT_KOHESIO"})
piv["MONTANT_TOTAL_INDICATIF"]=piv["MONTANT_FTS"]+piv["MONTANT_KOHESIO"]

# — répartition ANNUELLE (part de chaque canal cette année-là) —
tot=piv["MONTANT_TOTAL_INDICATIF"].replace(0,np.nan)
piv["PART_DIRECTE_ANNEE"] =(piv["MONTANT_FTS"]/tot*100).round(2)
piv["PART_PARTAGEE_ANNEE"]=(piv["MONTANT_KOHESIO"]/tot*100).round(2)

# — répartition sur TOUTE LA PÉRIODE (plus robuste) —
per=piv.groupby("SIREN")[["MONTANT_FTS","MONTANT_KOHESIO"]].sum()
per["TOT"]=per.sum(axis=1).replace(0,np.nan)
piv["MONTANT_FTS_PERIODE"]     =piv["SIREN"].map(per["MONTANT_FTS"])
piv["MONTANT_KOHESIO_PERIODE"] =piv["SIREN"].map(per["MONTANT_KOHESIO"])
piv["PART_DIRECTE_PERIODE"]    =piv["SIREN"].map((per["MONTANT_FTS"]/per["TOT"]*100).round(2))
piv["PART_PARTAGEE_PERIODE"]   =piv["SIREN"].map((per["MONTANT_KOHESIO"]/per["TOT"]*100).round(2))

def presence(a,b):
    return np.select([(a>0)&(b>0),a>0,b>0],["Les deux","FTS seul","Kohesio seul"],default="—")
piv["PRESENCE_ANNEE"]  =presence(piv["MONTANT_FTS"],piv["MONTANT_KOHESIO"])
piv["PRESENCE_PERIODE"]=presence(piv["MONTANT_FTS_PERIODE"],piv["MONTANT_KOHESIO_PERIODE"])

_n_avant  = len(piv)
_ent_avant = piv["SIREN"].nunique()
print(f"Avant filtrage : {_n_avant} couples SIREN × année | {_ent_avant} entités")
print(piv.drop_duplicates("SIREN")["PRESENCE_PERIODE"].value_counts().to_string())



## 5 · Assemblage final et export

In [ ]:
attrs=(empile.replace("",np.nan).groupby("SIREN")[[a for a in ATTRIBUTS if a in empile.columns]]
       .first().reset_index())
final=(piv.merge(lu_nom,on="SIREN",how="left").merge(attrs,on="SIREN",how="left"))

# ══════════════════════════════════════════════════════════════════════════
#  FEUILLE 1 — LARGE : 1 ligne par SIREN × année, colonnes essentielles
#  (pour tableaux, tris, et métriques simples)
# ══════════════════════════════════════════════════════════════════════════
COLS_ESSENTIELLES = ["NOM_API","ANNEE","MONTANT_FTS","MONTANT_KOHESIO","MONTANT_TOTAL"]

large = final.rename(columns={"NOM_REFERENCE":"NOM_API",
                              "MONTANT_TOTAL_INDICATIF":"MONTANT_TOTAL"})[
        ["SIREN","NOM_API","SOURCE_NOM_REFERENCE","ANNEE",
         "MONTANT_FTS","MONTANT_KOHESIO","MONTANT_TOTAL",
         "PRESENCE_ANNEE","PRESENCE_PERIODE",
         "PART_DIRECTE_PERIODE","PART_PARTAGEE_PERIODE"]].copy()

# Contrôle : d'où vient le nom retenu ?
_src = large.drop_duplicates("SIREN")["SOURCE_NOM_REFERENCE"].value_counts()
print("Origine du NOM_API retenu :")
for k,v in _src.items(): print(f"   {k:26s} {v:5d} entité(s)")
if "SIRENE (Nom_API)" not in _src.index:
    print("   ⚠️ Aucun nom issu de SIRENE : vérifiez COL_NOM_API_FTS / _KOH en cellule 0.")
large["SIREN"]=large["SIREN"].astype(str).str.zfill(9)
large=large.sort_values(["NOM_API","ANNEE"])

# ══════════════════════════════════════════════════════════════════════════
#  FEUILLE 2 — LONG : 1 ligne par SIREN × année × CANAL
#  ⚠️ C'EST CE FORMAT QU'IL FAUT POUR UN DIAGRAMME À BARRES EMPILÉES
# ══════════════════════════════════════════════════════════════════════════
long = large.melt(id_vars=["SIREN","NOM_API","ANNEE"],
                  value_vars=["MONTANT_FTS","MONTANT_KOHESIO"],
                  var_name="CANAL", value_name="MONTANT")
long["CANAL"]=long["CANAL"].map({"MONTANT_FTS":"Gestion directe",
                                 "MONTANT_KOHESIO":"Gestion partagée"})
long=long[long["MONTANT"]>0].sort_values(["NOM_API","ANNEE","CANAL"])

with pd.ExcelWriter(FICHIER_SORTIE, engine="openpyxl") as xl:
    long.to_excel(xl, sheet_name="POUR_GRAPHIQUE_long", index=False)
    large.to_excel(xl, sheet_name="POUR_TABLEAU_large", index=False)

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
wb=load_workbook(FICHIER_SORTIE)
for ws in wb.worksheets:
    for c in ws[1]: c.font=Font(bold=True); c.alignment=Alignment(vertical="center")
    for r in ws.iter_rows(min_col=1,max_col=1,min_row=2):    # SIREN en TEXTE
        for cell in r:
            if cell.value is not None:
                cell.value=str(cell.value).zfill(9); cell.number_format="@"
    ws.freeze_panes="B2"
wb.save(FICHIER_SORTIE)

print(f"✅ {FICHIER_SORTIE}")
print(f"   POUR_GRAPHIQUE_long : {len(long):5d} lignes — SIREN, NOM_API, ANNEE, CANAL, MONTANT")
print(f"   POUR_TABLEAU_large  : {len(large):5d} lignes — les 2 montants + total en colonnes")
print("\nAperçu format long (celui du graphique) :")
print(long.head(6).to_string(index=False))
try:
    from google.colab import files as _cf; _cf.download(FICHIER_SORTIE)
except Exception: pass


# ══ CONTRÔLE DE VRAISEMBLANCE ═══════════════════════════════════════════════
SEUIL_ALERTE = 100_000_000     # 100 M€ pour une entité sur une année
_susp = large[large["MONTANT_TOTAL"] > SEUIL_ALERTE]
if len(_susp):
    print(f"\n⚠️  {len(_susp)} ligne(s) au-dessus de {SEUIL_ALERTE:,.0f} € "
          f"— à vérifier (format de nombre ? doublon ?) :".replace(","," "))
    print(_susp.nlargest(10,"MONTANT_TOTAL")[
          ["NOM_API","ANNEE","MONTANT_FTS","MONTANT_KOHESIO","MONTANT_TOTAL"]].to_string(index=False))
else:
    print(f"\n✔ Contrôle de vraisemblance : aucun montant > {SEUIL_ALERTE:,.0f} €".replace(","," "))
print(f"   Montant maximum observé : {large['MONTANT_TOTAL'].max():,.0f} €".replace(","," "))


## 6 · Dans MicroStrategy

Importez ce fichier unique (Ajouter des données → Fichier). Puis :

- **Attributs** : `NOM_REFERENCE` (filtre bénéficiaire), `ANNEE` (filtre année),
  `REFERENTIEL_SGAE`, `OPERATEUR_ETAT`, `SECTION_NAF` (filtres complémentaires)
- **Métriques** : `MONTANT_FTS`, `MONTANT_KOHESIO`, et les parts en pourcentage

**Visuel suggéré** — barres empilées à 100 % : `NOM_REFERENCE` en axe,
`ANNEE` en découpage, `PART_DIRECTE_ANNEE` / `PART_PARTAGEE_ANNEE` en valeurs.
On lit d'un coup d'œil la bascule entre les deux canaux.

**Important** : ne créez pas de métrique additionnant `MONTANT_FTS` et
`MONTANT_KOHESIO` autrement que via `MONTANT_TOTAL_INDICATIF`, déjà étiquetée
comme telle. Vérifiez aussi que `SIREN` est bien reconnu en **texte** à l'import.


---

### Note sur les assiettes retenues

| Canal | Colonne | Ce qu'elle mesure |
|---|---|---|
| Gestion directe (FTS) | `Beneficiary contracted amount` | montant contractualisé par l'UE avec le bénéficiaire |
| Gestion partagée (Kohesio) | `Project_EU_Budget` | **part européenne** du projet |

Les deux assiettes sont **homogènes** : elles mesurent l'une comme l'autre une
contribution de l'Union européenne. Le total et la répartition entre canaux
répondent donc bien à la question « que capte cette entité de l'Europe, et par
quel canal ». C'est le choix à privilégier pour le pilotage de la captation.

À noter que `Project_EU_Budget` est un montant **programmé** sur toute la durée
du projet, rattaché à son année de démarrage, tandis que le montant FTS est
contractualisé sur un exercice. La comparaison reste donc pertinente en volume
et en répartition, mais l'annualisation est une convention (voir plus haut).

Pour mesurer l'investissement total mobilisé (cofinancement national compris),
basculer `KOH["montant"]` sur `Total_Eligible_Expenditure_amount` en cellule 0.


## 6 · Simulation du diagramme à barres

Aperçu de ce que donnera le graphique dans MicroStrategy : pour chaque
bénéficiaire, une barre par année, **empilée** en gestion directe / partagée.
On visualise ici les 8 plus gros bénéficiaires pour rester lisible.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TOP_N = 8
top = (large.groupby("NOM_API")["MONTANT_TOTAL"].sum()
       .sort_values(ascending=False).head(TOP_N).index.tolist())
d = large[large["NOM_API"].isin(top)].copy()
d["LIB"] = d["NOM_API"].str.slice(0,28) + " " + d["ANNEE"].astype(str)
d = d.sort_values(["NOM_API","ANNEE"])

fig, ax = plt.subplots(figsize=(11, max(4, 0.32*len(d))))
y = range(len(d))
ax.barh(y, d["MONTANT_FTS"],   label="Gestion directe (FTS)",     color="#4472C4")
ax.barh(y, d["MONTANT_KOHESIO"], left=d["MONTANT_FTS"],
        label="Gestion partagée (Kohesio)", color="#ED7D31")
ax.set_yticks(list(y)); ax.set_yticklabels(d["LIB"], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Montant (€)"); ax.legend(loc="lower right", fontsize=8)
ax.set_title(f"Répartition par canal — top {TOP_N} bénéficiaires, par année", fontsize=11)
ax.grid(axis="x", alpha=.3)
plt.tight_layout(); plt.savefig("simulation_graphique.png", dpi=110)
print("✅ simulation_graphique.png")
plt.show()

print("\n--- Équivalent MicroStrategy (feuille POUR_GRAPHIQUE_long) ---")
print("  Type de visuel : Barres empilées horizontales")
print("  Axe vertical (catégories) : NOM_API, puis ANNEE")
print("  Découpage / Couleur       : CANAL")
print("  Mesure                    : Sum(MONTANT)")
print("  Filtres                   : NOM_API (liste), ANNEE (multi-sélection)")
print("\n  Pour une lecture en % : même visuel en « barres empilées 100 % ».")
